<a href="https://colab.research.google.com/github/raoankit72005/minigpt/blob/main/GPT2_Pretrained_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OpenAI GPT-2 pretrained weights

This notebook loads GPT-2 small (124M) weights into the custom PyTorch implementation in this repository. It uses the original GPT-2 BPE tokenizer. The original character-level training notebook remains a separate experiment.

Reference: [Raschka’s weight-loading chapter](https://github.com/rasbt/LLMs-from-scratch/tree/main/ch05/02_alternative_weight_loading). GPT-2 generates text continuations; it is not an instruction-tuned chatbot.


In [1]:
!git clone https://github.com/raoankit72005/minigpt.git /content/minigpt
%cd /content/minigpt
%pip install -r requirements.txt


Cloning into '/content/minigpt'...
remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 17 (delta 2), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (17/17), 19.07 KiB | 1.73 MiB/s, done.
Resolving deltas: 100% (2/2), done.
/content/minigpt
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 55.6 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.31.0
    Uninstalling huggingface_hub-1.31.0:
      Successfully uninstalled huggingface_hub-1.31.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.2
    Uninstalling tokenizers-0.23.2:
      Successf

## Load and verify
Choose a GPU runtime for faster generation; CPU also works. The first run downloads roughly 500 MB of model weights plus the tokenizer. No paid API key is needed. The loader verifies its logits against the downloaded reference before releasing that model.


In [2]:
import torch
from gpt2_pretrained import load_pretrained
device = "cuda" if torch.cuda.is_available() else "cpu"
model, tokenizer = load_pretrained(cache_dir="checkpoints")
model = model.to(device)
print("Device:", device)
print("Parameters:", sum(p.numel() for p in model.parameters()))


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

PASS: mapped pretrained logits match the reference model.


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device: cpu
Parameters: 124439808


## Generate text
Change PROMPT and run this cell again. Sampling uses temperature and top-k; the full prompt plus its continuation is printed.


In [6]:
PROMPT = "Artificial intelligence will change education by"
ids = tokenizer.encode(PROMPT, return_tensors="pt").to(device)
if ids.shape[1] == 0:
    ids = torch.tensor([[tokenizer.eos_token_id]], device=device)
torch.manual_seed(42)
output = model.generate(ids, max_new_tokens=100, temperature=0.8, top_k=40, eos_id=tokenizer.eos_token_id)
print(tokenizer.decode(output[0].tolist(), skip_special_tokens=True))


Artificial intelligence will change education by transforming our lives and create new opportunities for more people to achieve their goals and achieve their goals.

For more information on the research that we are doing and what you can do to improve your educational outcomes, visit this website or follow us on Twitter and Facebook.

About the Foundation :

The Foundation is the leading research organization in the field of AI and artificial intelligence. Founded in 1997, our mission is to empower the next generation of researchers to pursue their dreams in an innovative way


## Optional: save converted weights
The Hugging Face cache already retains the downloaded weights for the lifetime of this runtime. This cell saves a portable custom checkpoint. Colab storage is temporary; copy the checkpoint and tokenizer to Drive if you want to retain them. Do not commit these large files to Git.


In [4]:
from dataclasses import asdict
from pathlib import Path
OUT = Path("checkpoints/converted_gpt2")
OUT.mkdir(parents=True, exist_ok=True)
torch.save({"config": asdict(model.cfg), "model_id": "openai-community/gpt2",
            "state_dict": {k: v.cpu() for k, v in model.state_dict().items()}}, OUT / "gpt2.pt")
tokenizer.save_pretrained(OUT)
print("Saved:", OUT)


Saved: checkpoints/converted_gpt2


## Optional: restore the converted checkpoint
Run after imports/installation when the saved directory is available.


In [5]:
from gpt2_pretrained import GPT2, GPT2Config
from transformers import GPT2TokenizerFast
checkpoint = torch.load(OUT / "gpt2.pt", map_location="cpu", weights_only=True)
restored = GPT2(GPT2Config(**checkpoint["config"]))
restored.load_state_dict(checkpoint["state_dict"], strict=True)
restored = restored.to(device).eval()
restored_tokenizer = GPT2TokenizerFast.from_pretrained(OUT)
